# 10.07 · Prompt management with the Prompt & Context Hub

**Module 10 · Testing, Evaluation & Observability** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1.5–2 h |
| 📋 Prerequisites | Lessons 10.03 and 10.05 (the release suite); Lesson 01.08 (`run_metadata`). |
| 🔑 Keys | none (optional: `LANGSMITH_API_KEY` for the 🌐 cell) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-15 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Treat** prompts as versioned data: commits identified by content, tags like `prod` that move, code that pins a tag or commit.
2. **Link** every trace and experiment to the exact prompt commit that produced it.
3. **Promote** a prompt version only through the eval gate, and see a plausible rewrite get blocked.
4. **Push and pull** prompts with the LangSmith SDK, and know what the Context Hub adds (versioned agents and skills).

## 📚 Official reading (verified 2026-10-15)

| Page | Read it for |
|---|---|
| [Prompt & Context Hub](https://docs.langchain.com/langsmith/prompt-context-hub) | Prompts (message templates) vs contexts (versioned agent/skill bundles promoted to environments) |
| [Manage prompts programmatically](https://docs.langchain.com/langsmith/manage-prompts-programmatically) | `push_prompt`, `pull_prompt("name:commit-or-tag")`, the in-memory prompt cache (5-minute TTL by default) |
| [Manage prompts](https://docs.langchain.com/langsmith/manage-prompts) | Tags, commits, webhooks |
| [Manage contexts with the SDK](https://docs.langchain.com/langsmith/manage-contexts-sdk) | `push_agent`, `push_skill`, `pull_agent("name:production")` (langsmith ≥ 0.7.35) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langsmith>=0.14.7,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langsmith", "langchain-core",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/10-testing-evaluation-observability/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm10-prompt-hub'
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m10-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · Why prompts live outside the code

Prompts change more often than code, and often by people who don't deploy code (support leads, domain experts). Storing them as versioned data gives you:

| Concept | Meaning | Why it matters |
|---|---|---|
| **Commit** | One immutable version of a prompt, identified by a hash | Traces and experiments point at exactly what ran |
| **Tag** | A movable name (`prod`, `staging`) pointing at a commit | Code pulls `triage:prod`; moving the tag is the release |
| **Pin** | Code that pulls a tag or a commit, never "whatever is latest" | Nobody changes production by saving a draft |
| **Promotion** | Moving `prod` to a new commit | Should happen only after the eval gate passes |

The risk is the flip side of the benefit: anyone with edit rights can change production behaviour without a code review. The gate is what keeps that safe.

---
## Part 2 · A prompt registry, offline

`PromptRegistry` is an offline twin of the Prompt Hub: `push` returns a commit id (a content hash: the same text is the same commit), `tag` moves a name, `pull` resolves `latest`, a tag or a commit.

In [ ]:
# 🧪 Offline: two versions of the triage prompt, a prod tag, and the link to trace metadata.
from mastery.agents.support import TRIAGE_PROMPT
from mastery.evals import PromptRegistry, run_metadata

registry = PromptRegistry()
v1 = registry.push("orbita-triage", TRIAGE_PROMPT, description="Module 09 triage prompt")
registry.tag("orbita-triage", v1, "prod")

FRIENDLY = ("You are Orbita Cloud's front desk. Greet the customer warmly and help them with whatever they need, "
            "in a friendly, concise tone.")
v2 = registry.push("orbita-triage", FRIENDLY, description="Friendlier tone (support lead's draft)")

for c in registry.history("orbita-triage"):
    print(c["commit"], "|", c["description"])
print("\nprod   ->", registry.resolve("orbita-triage", "prod"))
print("latest ->", registry.resolve("orbita-triage", "latest"))
print("same text, same commit:", registry.push("orbita-triage", TRIAGE_PROMPT) == v1)
meta = run_metadata(prompt_version=f"orbita-triage:{v1}", prompt=registry.pull("orbita-triage", "prod"))
print("trace metadata:", {k: meta[k] for k in ("prompt_version", "prompt_hash")})

**You should now see** two commits (the Module 09 prompt and the friendlier draft), `prod` on the first and `latest` on the second, `same text, same commit: True`, and trace metadata whose `prompt_hash` starts with the prod commit id.

### What just happened?

"Latest" and "prod" are already different: the draft exists, but production code that pulls `orbita-triage:prod` is unaffected. The commit id **is** the content hash, so the hash in every trace (Lesson 10.02) tells you exactly which version answered.

---
## Part 3 · Promotion goes through the gate

The friendlier draft reads well. Is it better? The release gate from Lesson 10.08 (the frozen release suite, the red-team attacks, a cost budget, thresholds from the last approved release) answers before `prod` moves. `Candidate(build=...)` builds the desk with the draft as triage prompt.

In [ ]:
# 🧪 Offline: gate both versions; promote() refuses a commit whose gate failed.
from functools import partial

from mastery.agents.release_suite import Candidate
from mastery.agents.support import build_handoffs_desk
from mastery.evals import PromotionBlocked
from mastery.evals.release_gate import evaluate_release, load_baseline

baseline = load_baseline()

def gate_for(commit):
    prompt = registry.pull("orbita-triage", commit)
    candidate = Candidate(f"triage@{commit}", build=partial(build_handoffs_desk, triage_prompt=prompt))
    report, _ = evaluate_release(candidate, baseline)
    return report

for commit in (v1, v2):
    report = gate_for(commit)
    print(f"--- orbita-triage@{commit}")
    print(report.to_markdown())
    try:
        registry.promote("orbita-triage", commit, "prod", report)
        print("-> promoted to prod")
    except PromotionBlocked as blocked:
        print("->", blocked)
print("\nprod is still", registry.resolve("orbita-triage", "prod"))

**You should now see** the Module 09 prompt pass all four gates (it's the approved baseline) and get re-promoted (a no-op), then the friendly draft **fail** the pass-rate and pass^3 gates (around 11% of cases pass: the triage agent no longer transfers), with the red-team and cost gates still green, and `PromotionBlocked`. `prod` stays on the first commit.

### What just happened?

The draft dropped one instruction ("transfer the customer to the right specialist with a transfer tool"), and the desk stopped routing. Nothing crashed and the answers stayed polite, so a manual read of three conversations might have approved it. With the gate, the release mechanism itself refuses. (The red team still passes because a desk that does nothing can't be tricked into doing something wrong: one more reason no single gate is enough.)

> ⚠️ **Production note: pin prompts and promote only through the gate.**
> Production code pulls a tag or commit (`name:prod`), never latest; moving the tag is a release and goes through the same eval gate as code; record the prompt commit in every trace. The SDK caches pulled prompts in memory (5-minute TTL by default), so a tag move reaches running processes within minutes, not instantly; for offline or air-gapped services, ship a dumped cache file.

---
## Part 4 · The same flow in LangSmith

> 🌐 Talks to LangSmith (free Developer plan is enough; no model tokens). Without `LANGSMITH_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 🌐 LangSmith: push both versions, tag the first as prod, pull by tag and by commit.
if HAVE_LANGSMITH_KEY:
    import uuid

    from langchain_core.prompts import ChatPromptTemplate
    from langsmith import Client

    client = Client()
    name = f"m10-orbita-triage-{uuid.uuid4().hex[:6]}"
    url_v1 = client.push_prompt(name, object=ChatPromptTemplate.from_messages([("system", TRIAGE_PROMPT), ("human", "{message}")]),
                                description="Orbita triage (course)", commit_tags=["prod"])
    url_v2 = client.push_prompt(name, object=ChatPromptTemplate.from_messages([("system", FRIENDLY), ("human", "{message}")]))
    prod = client.pull_prompt(f"{name}:prod")
    print("prod system prompt starts:", prod.messages[0].prompt.template[:50])
    print("latest url:", url_v2)
else:
    print("⏭️ Skipped: LANGSMITH_API_KEY is not set (see the setup cell).")

**You should now see** `⏭️ Skipped` without a key. With one, the prod prompt starting with *"You are Orbita Cloud's triage agent..."* even though the latest commit is the friendly draft. Delete the test prompt afterwards in the UI (or `client.delete_prompt(name)`).

---
## Part 5 · Contexts: versioning more than one prompt

A desk is more than one prompt: triage plus three specialists, tool descriptions, the refund policy. The **Context Hub** versions such bundles as **agent repos** and **skill repos** (files like `AGENTS.md`, `SKILL.md`, `tools.json`), promoted to **environments** so each deployment pulls the right version. With the SDK (langsmith ≥ 0.7.35; not run here):

```python
from langsmith import Client
from langsmith.schemas import FileEntry

client = Client()
client.push_agent("orbita-desk", files={
    "AGENTS.md": FileEntry(content=TRIAGE_PROMPT),
    "skills/billing/SKILL.md": FileEntry(content=billing_instructions),
}, description="Orbita support desk")
prod_desk = client.pull_agent("orbita-desk:production")   # a tag, or a commit hash
```

The rule doesn't change with the size of the bundle: a new context version goes through the release gate before its environment tag moves. Module 11 builds deep agents whose instructions and skills are exactly this kind of bundle.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Production changed and nobody deployed | Code pulls `latest` | Pull `name:prod` (or a commit); tag moves are releases |
| Trace says "triage-v1" but behaviour changed | Prompt edited without bumping the name | Record the commit/hash (`run_metadata(prompt=...)`), not only a human label |
| Tag moved, service still uses the old prompt | SDK prompt cache (5-minute TTL) | Wait, `skip_cache=True`, or restart; plan for the delay |
| A "harmless" rewording broke routing | An instruction the agent relied on was removed | Gate every prompt change on the release suite |
| `push_agent` missing | langsmith older than 0.7.35 | Upgrade the SDK |

## ⚠️ Production notes

> ⚠️ **Production note: pin prompts and promote only through the gate.** (Part 3)

## 🏋️ Exercises

1. **(Easy)** Push a third version that adds one sentence to the Module 09 prompt ("Customer IDs look like C-1001."). Is it a new commit? Does it pass the gate?
2. **(Medium)** Save the registry to `OUT / "prompts.json"`, load it in a new `PromptRegistry`, and show that `prod` resolves to the same commit.
3. **(Medium)** Write a `rollback(registry, name, tag)` that moves a tag back to the previous commit it pointed at. What extra state do you need to keep?
4. **(Stretch)** Make the friendly draft pass: keep its tone but restore the routing instruction. Gate it and promote it to a `staging` tag.

<details>
<summary><b>Show solutions</b></summary>

**1.** A new commit (different text, different hash). Offline it passes with the same numbers as v1: the rule-based model ignores the extra sentence. With a live model it might help lowercase IDs; that's what the gate on a live run would tell you.
```python
v3 = registry.push("orbita-triage", TRIAGE_PROMPT + " Customer IDs look like C-1001.")
print(v3, gate_for(v3).passed)
```

**2.**
```python
path = registry.save(OUT / "prompts.json")
again = PromptRegistry.load(path)
print(again.resolve("orbita-triage", "prod") == registry.resolve("orbita-triage", "prod"))
```
(`OUT` comes from the setup of the earlier lessons; define `OUT = REPO_ROOT / "data" / "outputs" / "m10"` if you run this alone.)

**3.** The registry stores only where a tag points **now**. Keep a log of moves, e.g. `moves = [(name, tag, old, new)]` appended inside your own `promote` wrapper; `rollback` then re-tags `old` (and should still record the move). LangSmith keeps commit history; the tag history is what you need to log yourself or read from webhooks.

**4.**
```python
friendly_routed = ("You are Orbita Cloud's triage agent. Greet the customer warmly, in a friendly, concise tone. "
                   "Do not solve problems yourself: transfer the customer to the right specialist with a transfer tool "
                   "(billing, technical or account).")
v4 = registry.push("orbita-triage", friendly_routed, description="friendly + routing")
report = gate_for(v4)
registry.promote("orbita-triage", v4, "staging", report)
print(report.to_markdown())
```
It passes once the prompt says **both** what the agent is (the triage role) and what it must do (transfer). Adding only the transfer sentence to the front-desk draft still fails offline: the agent's role comes from the first sentence. A live model is more forgiving, but the lesson holds: keep the instructions behaviour depends on, and let the gate tell you.
</details>

## 🔎 Docs drill (5 min)

In the Prompt & Context Hub, what is the difference between a **prompt** and a **context**?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Prompt & Context Hub](https://docs.langchain.com/langsmith/prompt-context-hub).
2. A **prompt** is an individual message template you send to a model.
3. A **context** is a versioned bundle of instructions and tools that defines a skill or a full agent, promoted through environments so agents pull the right version.

</details>

## ✅ Checkpoint

- [ ] My code pulls prompts by tag or commit, never latest.
- [ ] Every trace records the prompt commit (or content hash) that produced it.
- [ ] Moving `prod` goes through the release gate, and I saw a plausible rewrite get blocked.
- [ ] I know what the Context Hub versions beyond single prompts.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check [Prompt & Context Hub](https://docs.langchain.com/langsmith/prompt-context-hub) and [Manage contexts with the SDK](https://docs.langchain.com/langsmith/manage-contexts-sdk) and log in [`docs/research-log.md`](../../docs/research-log.md): the minimum SDK version for contexts (2026-10-15: 0.7.35) and the default prompt-cache TTL (300 s).

## Summary

- Prompts are versioned data: commits by content, movable tags, code that pins.
- The commit id in every trace says exactly which version answered.
- Promotion is a release: it goes through the eval gate, and the gate catches rewrites that read well and break behaviour.
- The Context Hub versions whole agents and skills the same way.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 10-08-testing-and-ci-gates.ipynb** → [`10-08-testing-and-ci-gates.ipynb`](10-08-testing-and-ci-gates.ipynb).

[Module 10 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)